<div align="center">

<h1 style="color:blue; font-weight:bold;">
Data Integration (HadUK-Grid Weather and DfT Road Collisions)
</h1>

</div>

This notebook combines the two cleaned datasets produced in notebooks 01 and 02:

- **Weather:** daily rainfall and mean air temperature on a 5 km grid for 2025 (HadUK-Grid), masked to UK land cells.
- **Collisions:** 101,523 road collisions in Great Britain in 2025 (DfT), one row per collision.

Each collision is assigned the rainfall and temperature recorded in its 5 km grid cell on the day it happened. Because both datasets use British National Grid coordinates (EPSG:27700), the match is made by arithmetic on the coordinates, with no reprojection.

The notebook also produces a daily summary for Great Britain, giving the number of collisions alongside the rainfall and temperature on each day. This allows the analysis to compare collisions per day under different weather, rather than raw totals that depend on how many days of each kind of weather occurred.

### Step 1: Import Required Libraries

Import the libraries required to load the two cleaned datasets and combine them. `xarray` reads the gridded weather data from NetCDF, `pandas` handles the collision table, `numpy` supports the grid-index arithmetic, and `pyproj` confirms the coordinate reference system stored in the weather file.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import CRS

### Step 2: Load and Verify the Cleaned Datasets

Load the cleaned weather dataset (notebook 01) and the cleaned collision dataset (notebook 02), and confirm that they are ready to combine.

We check that:
- the weather file contains rainfall, temperature, the analysis mask and its coordinate reference system,
- both datasets cover the same 365 days of 2025,
- every collision has a location and a unique identifier.

The CSV file does not store column types, so they are specified again on loading, as in notebook 02's reload check.

In [2]:
# Paths to the outputs of notebooks 01 and 02.
weather_path = Path("../data/weather/processed/uk_weather_2025_masked.nc")
collision_path = Path("../data/incident/processed/collisions_clean_2025.csv")

# Open the weather dataset. xarray reads values from disk only when needed,
# so opening the file is quick even though it holds 365 daily grids.
weather = xr.open_dataset(weather_path)

# Reload the collisions with the same column types used in notebook 02.
flag_columns = ["police_reported_rain", "police_reported_snow", "surface_wet", "surface_ice_snow"]
collisions = pd.read_csv(
    collision_path,
    dtype={
        "collision_index": str,
        "local_authority_ons_district": str,
        **{column: "boolean" for column in flag_columns},
    },
    parse_dates=["date", "datetime"],
)

# --- Weather dataset ---

print("Weather variables:", list(weather.data_vars))
print("Weather dimensions:", dict(weather.sizes))
print("Weather period:", str(weather.time.values[0])[:10], "to", str(weather.time.values[-1])[:10])

# The analysis mask saved in notebook 01 marks the retained cells (1 = keep).
analysis_mask = weather["analysis_mask"].astype(bool).values
print("Retained weather cells:", int(analysis_mask.sum()))

# Confirm the CRS travelled with the file.
weather_crs = CRS.from_cf(weather["transverse_mercator"].attrs)
print("Weather CRS method:", weather_crs.coordinate_operation.method_name)

# --- Collision dataset ---

print("\nCollisions:", collisions.shape)
print("Collision period:", collisions["date"].min().date(), "to", collisions["date"].max().date())
print("Collision date type:", collisions["date"].dtype)

# --- Readiness checks ---

assert weather.sizes["time"] == 365
assert collisions["collision_index"].is_unique
assert collisions["location_easting_osgr"].notna().all()

# HadUK-Grid timestamps each day at 12:00, so compare calendar dates only.
weather_days = pd.DatetimeIndex(weather.time.values).normalize()
collision_days = pd.DatetimeIndex(collisions["date"].unique()).normalize()
print("Collision dates missing from the weather data:",
      len(collision_days.difference(weather_days)))

Weather variables: ['rainfall', 'tas', 'analysis_mask', 'coastal_cell', 'transverse_mercator']
Weather dimensions: {'time': 365, 'projection_y_coordinate': 290, 'projection_x_coordinate': 180}
Weather period: 2025-01-01 to 2025-12-31
Retained weather cells: 10400
Weather CRS method: Transverse Mercator

Collisions: (101523, 31)
Collision period: 2025-01-01 to 2025-12-31
Collision date type: datetime64[ns]
Collision dates missing from the weather data: 0


### Step 3: Restrict the Weather Grid to Great Britain

The DfT collision data covers Great Britain only (England, Wales and Scotland), but the weather mask from notebook 01 also includes Northern Ireland. Collisions will never be matched to Northern Ireland cells, but those cells would still distort any Great Britain-wide weather figure, such as the average rainfall on each day used in Step 7.

We therefore build a Great Britain mask using the same ONS boundary file and the same cell-overlap test as notebook 01, this time leaving out Northern Ireland. A cell is kept if it is in the notebook 01 analysis mask and any part of it overlaps England, Wales or Scotland.

In [3]:
import geopandas as gpd
import pyogrio
import shapely

# Load the same ONS boundary used in notebook 01.
boundary_path = Path("../data/boundaries/Countries_December_2025_Boundaries_UK_BFC_-2912039989678096674.geojson")

# Remove GDAL's default GeoJSON size limit, as in notebook 01.
pyogrio.set_gdal_config_options({"OGR_GEOJSON_MAX_OBJ_SIZE": "0"})
uk_boundary = gpd.read_file(boundary_path)
print("Countries in the boundary file:", uk_boundary["CTRY25NM"].tolist())

# Keep England, Wales and Scotland, and combine them into one geometry.
gb_boundary = uk_boundary[uk_boundary["CTRY25NM"] != "Northern Ireland"]
assert len(gb_boundary) == 3, "Expected England, Scotland and Wales."
gb_geometry = gb_boundary.geometry.union_all()

# Preparing the geometry speeds up the many overlap tests below.
shapely.prepare(gb_geometry)

# Build a 5 km square for every grid cell, as in notebook 01 Step 8.5.
grid_x = weather["projection_x_coordinate"].values
grid_y = weather["projection_y_coordinate"].values
cell_size = float(grid_x[1] - grid_x[0])
half_cell = cell_size / 2

x_mesh, y_mesh = np.meshgrid(grid_x, grid_y)
cell_boxes = shapely.box(
    x_mesh - half_cell, y_mesh - half_cell,
    x_mesh + half_cell, y_mesh + half_cell,
)

# Great Britain cells: retained in notebook 01 and overlapping England, Wales or Scotland.
overlaps_gb = shapely.intersects(gb_geometry, cell_boxes)
gb_mask = analysis_mask & overlaps_gb
ni_cells = analysis_mask & ~overlaps_gb

print("\nRetained weather cells (UK):", int(analysis_mask.sum()))
print("Cells in Great Britain:", int(gb_mask.sum()))
print("Cells outside Great Britain (Northern Ireland):", int(ni_cells.sum()))

# Every retained cell must fall in exactly one of the two groups.
assert int(gb_mask.sum()) + int(ni_cells.sum()) == int(analysis_mask.sum())

# Apply the Great Britain mask to the weather variables.
gb_mask_da = xr.DataArray(
    gb_mask,
    coords={
        "projection_y_coordinate": weather["projection_y_coordinate"],
        "projection_x_coordinate": weather["projection_x_coordinate"],
    },
    dims=["projection_y_coordinate", "projection_x_coordinate"],
)
weather_gb = weather[["rainfall", "tas"]].where(gb_mask_da)

# The Great Britain cells should still have complete observations.
missing_in_gb = int(weather_gb["rainfall"].isnull().where(gb_mask_da, False).sum())
print("Missing rainfall values in GB cells:", missing_in_gb)
assert missing_in_gb == 0

Countries in the boundary file: ['England', 'Northern Ireland', 'Scotland', 'Wales']

Retained weather cells (UK): 10400
Cells in Great Britain: 9779
Cells outside Great Britain (Northern Ireland): 621
Missing rainfall values in GB cells: 0


#### Interpretation of Step 3

Of the 10,400 weather cells retained in notebook 01, **9,779 cells cover Great Britain** and 621 cover Northern Ireland. The Northern Ireland cells are excluded from the rest of the analysis because the DfT data has no collisions there. The Great Britain cells keep complete rainfall and temperature for all 365 days.

### Step 4: Assign Each Collision to a Weather Grid Cell

Each collision is placed in the 5 km grid cell that contains it, using its British National Grid easting and northing. Because the weather grid and the collisions use the same coordinate system, a cell's row and column follow directly from the coordinates.

Notebook 01 showed that 268 collisions fall in cells where HadUK-Grid has no data, mostly coastal slivers and small islands outside its land mask. For these, we use the nearest Great Britain cell with data, but only if its centre is within 10 km (two cells). Weather 10 km away is still representative of the same local conditions, while anything further would be guessing. Collisions with no cell within that distance are kept in the dataset but flagged as having no weather match, and are excluded from the weather analysis.

Each collision is labelled with how it was matched:
- **own cell**: the collision's own cell has weather data,
- **nearest cell**: weather taken from the nearest Great Britain cell within 10 km,
- **no match**: no Great Britain cell with data within 10 km.

In [4]:
# cKDTree finds the nearest point in a set quickly; used for collisions
# whose own cell has no weather data.
from scipy.spatial import cKDTree

# --- 1. Each collision's own grid cell ------------------------------------------

easting = collisions["location_easting_osgr"].to_numpy()
northing = collisions["location_northing_osgr"].to_numpy()

# Cell edges start half a cell before the first centre (e.g. -200,000 m),
# so dividing the distance from that edge by 5,000 m gives the column or row.
col_index = np.floor((easting - (grid_x[0] - half_cell)) / cell_size).astype(int)
row_index = np.floor((northing - (grid_y[0] - half_cell)) / cell_size).astype(int)

# Every collision should fall somewhere on the weather grid.
assert (
    (col_index >= 0) & (col_index < len(grid_x))
    & (row_index >= 0) & (row_index < len(grid_y))
).all()

in_gb_cell = gb_mask[row_index, col_index]
in_ni_cell = ni_cells[row_index, col_index]

print("Collisions in their own Great Britain weather cell:", int(in_gb_cell.sum()))
print("Collisions in a Northern Ireland cell:", int(in_ni_cell.sum()))
print("Collisions in a cell with no weather data:", int((~in_gb_cell).sum()))

# --- 2. Nearest Great Britain cell for the unmatched collisions ---------------

# Two cells (10 km) is the furthest we accept weather from another cell.
max_snap_distance = 2 * cell_size

# Centres of all Great Britain cells with weather data.
gb_rows, gb_cols = np.nonzero(gb_mask)
tree = cKDTree(np.column_stack([grid_x[gb_cols], grid_y[gb_rows]]))

unmatched = ~in_gb_cell
distance, nearest = tree.query(np.column_stack([easting[unmatched], northing[unmatched]]))
snap_ok = distance <= max_snap_distance

# Start from each collision's own cell, then replace it for the snapped ones.
match_row = row_index.copy()
match_col = col_index.copy()
idx_unmatched = np.flatnonzero(unmatched)
match_row[idx_unmatched[snap_ok]] = gb_rows[nearest[snap_ok]]
match_col[idx_unmatched[snap_ok]] = gb_cols[nearest[snap_ok]]

# --- 3. Record how each collision was matched ----------------------------------

weather_match = np.full(len(collisions), "own cell", dtype=object)
weather_match[idx_unmatched[snap_ok]] = "nearest cell"
weather_match[idx_unmatched[~snap_ok]] = "no match"
collisions["weather_match"] = weather_match

# Store the matched cell; -1 marks collisions with no weather match.
collisions["weather_cell_row"] = np.where(weather_match == "no match", -1, match_row)
collisions["weather_cell_col"] = np.where(weather_match == "no match", -1, match_col)

print(f"\nDistance from unmatched collisions to the nearest GB cell centre (m): "
      f"median {np.median(distance):,.0f}, max {distance.max():,.0f}")
print(collisions["weather_match"].value_counts().to_string())

# Where are the collisions that could not be matched?
lost = collisions.loc[collisions["weather_match"] == "no match"]
if len(lost):
    print("\nCollisions with no weather match, by country:")
    print(lost["country"].value_counts().to_string())

Collisions in their own Great Britain weather cell: 101255
Collisions in a Northern Ireland cell: 0
Collisions in a cell with no weather data: 268

Distance from unmatched collisions to the nearest GB cell centre (m): median 3,147, max 3,924
weather_match
own cell        101255
nearest cell       268


#### Interpretation of Step 4

| Match type | Collisions | Share |
|---|---:|---:|
| Own cell | 101,255 | 99.74% |
| Nearest cell (within 10 km) | 268 | 0.26% |
| No match | 0 | 0% |

No collision falls in a Northern Ireland cell, which confirms the coverage found in notebook 02.

The 268 collisions in cells without HadUK-Grid data are all between 3.1 km (median) and 3.9 km (maximum) from the centre of the nearest Great Britain cell with data. This is an adjacent cell, so the 10 km limit was never needed. These collisions are near the coast, where the 5 km cell containing them falls outside the HadUK-Grid land mask but the neighbouring cell does not. Taking weather from that neighbour is a small approximation compared with the 5 km resolution of the grid itself.

Every collision in the dataset can therefore be matched to weather data.

### Step 5: Attach Daily Rainfall and Temperature to Each Collision

Each collision now has a grid cell (row and column) and a date. We look up the rainfall and mean temperature for that cell on that day and add them as two new columns:
- `rainfall_mm`: total rainfall in the cell over the day (mm),
- `tas_c`: daily mean air temperature in the cell (°C), the mid-range of the day's maximum and minimum (see notebook 01, Step 4).

We also record whether the matched cell is one of the coastal cells added in notebook 01 (`coastal_cell`), so the analysis can check whether results change without them.

The weather grids are loaded into memory once, then all 101,523 values are picked out in a single indexing operation, which is far faster than looking up collisions one at a time.

In [5]:
# --- 1. Day index for each collision ---

# Map each calendar date to its position on the weather time axis (0 = 1 January).
# HadUK-Grid timestamps days at 12:00, so dates are compared without the time.
day_lookup = pd.Series(np.arange(len(weather_days)), index=weather_days)
time_index = day_lookup.reindex(pd.DatetimeIndex(collisions["date"]).normalize()).to_numpy()

# Every collision date must exist in the weather data (checked in Step 2).
assert not np.isnan(time_index).any()
time_index = time_index.astype(int)

# --- 2. Load the weather grids into memory ---

# Arrays shaped (day, row, column); Great Britain cells only, from Step 3.
rainfall_grid = weather_gb["rainfall"].values
tas_grid = weather_gb["tas"].values
coastal_grid = weather["coastal_cell"].values.astype(bool)

# --- 3. Look up each collision's weather ---

has_match = (collisions["weather_match"] != "no match").to_numpy()
r = collisions["weather_cell_row"].to_numpy()
c = collisions["weather_cell_col"].to_numpy()

# Collisions without a match keep NaN (none in this dataset, but kept general).
collisions["rainfall_mm"] = np.nan
collisions["tas_c"] = np.nan

# One indexing step picks [day, row, column] for every matched collision at once.
collisions.loc[has_match, "rainfall_mm"] = rainfall_grid[time_index[has_match], r[has_match], c[has_match]]
collisions.loc[has_match, "tas_c"] = tas_grid[time_index[has_match], r[has_match], c[has_match]]

# Flag collisions matched to a coastal cell (centre outside the boundary).
collisions["coastal_cell"] = False
collisions.loc[has_match, "coastal_cell"] = coastal_grid[r[has_match], c[has_match]]

# --- 4. Checks and summary ---

print("Collisions with weather attached:", int(collisions["rainfall_mm"].notna().sum()))
print("Missing rainfall among matched collisions:", int(collisions.loc[has_match, "rainfall_mm"].isna().sum()))
print("Missing temperature among matched collisions:", int(collisions.loc[has_match, "tas_c"].isna().sum()))
print("Collisions in coastal cells:", int(collisions["coastal_cell"].sum()))

assert collisions.loc[has_match, ["rainfall_mm", "tas_c"]].notna().all().all()

print()
print(collisions[["rainfall_mm", "tas_c"]].describe().round(2))
print("\nShare of collisions on days with no rain in their cell:",
      f"{100 * (collisions['rainfall_mm'] == 0).mean():.1f}%")

Collisions with weather attached: 101523
Missing rainfall among matched collisions: 0
Missing temperature among matched collisions: 0
Collisions in coastal cells: 3342

       rainfall_mm      tas_c
count    101523.00  101523.00
mean          1.90      11.73
std           4.36       5.62
min           0.00      -4.59
25%           0.00       7.25
50%           0.07      11.84
75%           1.59      16.03
max          81.47      28.01

Share of collisions on days with no rain in their cell: 10.3%


#### Interpretation of Step 5

All 101,523 collisions now have a daily rainfall and mean temperature value, with none missing. 3,342 collisions (3.3%) are matched to coastal cells.

On collision days, rainfall in the collision's cell had a median of 0.07 mm and a mean of 1.90 mm, with a maximum of 81.47 mm. Mean temperature ranged from -4.59 °C to 28.01 °C. Only 10.3% of collisions occurred with exactly 0 mm, because gridded rainfall is interpolated from rain gauges and often gives very small values on dry days. The analysis therefore uses standard thresholds rather than "any rainfall": a rain day is at least 0.2 mm, and the Met Office's "days of rain" are at least 1 mm.

#### Step 5.1: Check How HadUK-Grid Defines a Day

In British climatology, daily rainfall is usually measured over the 24 hours from 09:00 UTC, not from midnight. If HadUK-Grid follows this convention, a collision at 07:30 on 5 February falls within the rainfall total labelled 4 February, and matching on calendar date alone would assign it the wrong day.

The original monthly files store the start and end of each day in a variable called `time_bnds`. We inspect it for both rainfall and temperature before deciding whether the match in Step 5 needs adjusting.

In [6]:
# The processed file from notebook 01 does not keep time_bnds,
# so read it from the first original monthly file of each variable.
raw_rainfall_file = sorted(Path("../data/weather/rainfall").glob("*.nc"))[0]
raw_tas_file = sorted(Path("../data/weather/temperature").glob("*.nc"))[0]

for name, path, variable in [("Rainfall", raw_rainfall_file, "rainfall"),
                             ("Temperature", raw_tas_file, "tas")]:
    with xr.open_dataset(path) as raw:
        print(f"{name}: {path.name}")
        print("  First two time stamps:", raw["time"].values[:2])
        # Each row is [start, end] of the period the daily value covers.
        print("  First two time bounds:")
        print(raw["time_bnds"].values[:2])
        print("  cell_methods:", raw[variable].attrs.get("cell_methods"))
        print()

Rainfall: rainfall_hadukgrid_uk_5km_day_20250101-20250131.nc
  First two time stamps: ['2025-01-01T12:00:00.000000000' '2025-01-02T12:00:00.000000000']
  First two time bounds:
[['2025-01-01T09:00:00.000000000' '2025-01-02T09:00:00.000000000']
 ['2025-01-02T09:00:00.000000000' '2025-01-03T09:00:00.000000000']]
  cell_methods: time: sum

Temperature: tas_hadukgrid_uk_5km_day_20250101-20250131.nc
  First two time stamps: ['2025-01-01T12:00:00.000000000' '2025-01-02T12:00:00.000000000']
  First two time bounds:
[['2025-01-01T00:00:00.000000000' '2025-01-02T00:00:00.000000000']
 ['2025-01-02T00:00:00.000000000' '2025-01-03T00:00:00.000000000']]
  cell_methods: time: mid_range



#### Interpretation of Step 5.1

The two HadUK-Grid variables define a day differently:
- **Rainfall** totals run from **09:00 UTC to 09:00 UTC** the next day (the British climatological rain day). The value labelled 1 January covers 09:00 on 1 January to 09:00 on 2 January.
- **Temperature** runs from **midnight to midnight UTC**.

The calendar-date match in Step 5 is therefore wrong for rainfall whenever a collision happens before 09:00 UTC: that collision belongs to the previous day's rainfall total. DfT collision times are also in UK local time, which is one hour ahead of UTC during British Summer Time. Step 5.2 corrects both.

#### Step 5.2: Align Collision Times with the HadUK-Grid Day Definitions

Each collision's local time is converted to UTC, then:
- its **rainfall day** is the date of (UTC time minus 9 hours), so collisions before 09:00 UTC take the previous day's total,
- its **temperature day** is the UTC date.

The rainfall and temperature values from Step 5 are then replaced with the correctly aligned ones.

Collisions on 1 January before 09:00 UTC belong to the rain day starting on 31 December 2024, which is outside the downloaded data. Their rainfall is left blank rather than guessed.

The clock-change hours need a rule. When clocks go back in October, times between 01:00 and 02:00 occur twice, and these are treated as GMT. When clocks go forward in March, times between 01:00 and 02:00 do not exist, and any recorded there are moved forward to BST. This affects only a handful of collisions.

In [7]:
# --- 1. Convert collision times from UK local time to UTC -----------------------

# DfT times are local clock time (GMT in winter, BST = UTC+1 in summer).
# ambiguous=False: repeated hour when clocks go back is treated as GMT.
# nonexistent="shift_forward": skipped hour when clocks go forward moves to BST.
local_time = collisions["datetime"].dt.tz_localize(
    "Europe/London", ambiguous=False, nonexistent="shift_forward"
)
utc_time = local_time.dt.tz_convert("UTC").dt.tz_localize(None)

# --- 2. The HadUK-Grid day each collision belongs to ---

# Rainfall days run 09:00-09:00 UTC, so subtracting 9 hours gives the date
# of the rain day. Temperature days run midnight-midnight UTC.
rain_day = (utc_time - pd.Timedelta(hours=9)).dt.normalize()
tas_day = utc_time.dt.normalize()
calendar_day = collisions["date"].dt.normalize()

print("Collisions whose rainfall day differs from the calendar date:",
      int((rain_day != calendar_day).sum()))
print("Collisions whose temperature day differs from the calendar date:",
      int((tas_day != calendar_day).sum()))

# Position of each day on the weather time axis; NaN if outside 2025.
rain_index = day_lookup.reindex(rain_day).to_numpy()
tas_index = day_lookup.reindex(tas_day).to_numpy()

no_rain_day = np.isnan(rain_index)
no_tas_day = np.isnan(tas_index)
print("Collisions whose rainfall day falls outside 2025:", int(no_rain_day.sum()))
print("Collisions whose temperature day falls outside 2025:", int(no_tas_day.sum()))

# --- 3. Replace the Step 5 values with correctly aligned ones ---

ok_rain = has_match & ~no_rain_day
ok_tas = has_match & ~no_tas_day

# Keep the Step 5 values briefly, to measure how many change.
rainfall_before = collisions["rainfall_mm"].copy()

collisions["rainfall_mm"] = np.nan
collisions["tas_c"] = np.nan
collisions.loc[ok_rain, "rainfall_mm"] = rainfall_grid[rain_index[ok_rain].astype(int), r[ok_rain], c[ok_rain]]
collisions.loc[ok_tas, "tas_c"] = tas_grid[tas_index[ok_tas].astype(int), r[ok_tas], c[ok_tas]]

# Keep the rain day, since daily rainfall analysis must group by it.
collisions["rain_day"] = rain_day

changed = (collisions["rainfall_mm"] != rainfall_before) & collisions["rainfall_mm"].notna()
print("\nCollisions whose rainfall value changed:", int(changed.sum()))
print("Collisions with rainfall attached:", int(collisions["rainfall_mm"].notna().sum()))
print("Collisions with temperature attached:", int(collisions["tas_c"].notna().sum()))
print()
print(collisions[["rainfall_mm", "tas_c"]].describe().round(2))

Collisions whose rainfall day differs from the calendar date: 20889
Collisions whose temperature day differs from the calendar date: 898
Collisions whose rainfall day falls outside 2025: 55
Collisions whose temperature day falls outside 2025: 0

Collisions whose rainfall value changed: 19889
Collisions with rainfall attached: 101468
Collisions with temperature attached: 101523

       rainfall_mm      tas_c
count    101468.00  101523.00
mean          1.92      11.73
std           4.42       5.63
min           0.00      -4.59
25%           0.00       7.25
50%           0.07      11.84
75%           1.61      16.03
max          81.47      28.01


#### Interpretation of Step 5.2

| Check | Result |
|---|---:|
| Collisions moved to a different rainfall day | 20,889 (20.6%) |
| Collisions whose rainfall value changed | 19,889 |
| Collisions moved to a different temperature day | 898 |
| Collisions without rainfall (1 January before 09:00 UTC) | 55 |
| Collisions with rainfall attached | 101,468 |
| Collisions with temperature attached | 101,523 |

Matching on calendar date would have given about one collision in five the wrong day's rainfall: mainly collisions before 09:00, including the morning rush hour. The 898 temperature changes are collisions between midnight and 01:00 during British Summer Time, which fall on the previous UTC day.

The 55 collisions without rainfall are excluded from rainfall analysis only; they keep their temperature value. The summary statistics barely change (mean rainfall 1.90 to 1.92 mm), because the correction moves individual collisions between days rather than shifting the overall distribution.

### Step 6: Validate the Match Against Police-Recorded Conditions

The police record the weather and road surface at the scene of each collision. These records are independent of HadUK-Grid, so they provide a check on the match: if collisions have been paired with the right cell and day, then collisions where police reported rain should have more HadUK-Grid rainfall than those where they did not, and collisions on icy or snowy roads should have lower temperatures.

We compare:
- HadUK-Grid rainfall for collisions with and without police-reported rain, including the share falling on a rain day (at least 0.2 mm),
- the same comparison using the calendar-date match from Step 5, to measure whether the 09:00 alignment improved agreement,
- HadUK-Grid temperature for collisions with and without police-reported ice or snow.

Only collisions with a known police report are used (Step 8 of notebook 02 left unknown reports blank).

Agreement is not expected to be perfect. The police record conditions at one moment and place, while HadUK-Grid gives a 24-hour total averaged over a 5 km cell. Rain that fell in the early hours may have stopped by the time of an afternoon collision.

In [8]:
# Rain day threshold used throughout the analysis (British climatology).
rain_threshold = 0.2

# Collisions with both a known police weather report and HadUK-Grid rainfall.
known_rain = collisions["police_reported_rain"].notna() & collisions["rainfall_mm"].notna()
check = collisions.loc[known_rain].copy()

# The calendar-date rainfall from Step 5, kept for comparison.
check["rainfall_calendar_day"] = rainfall_before[known_rain]
police_rain = check["police_reported_rain"].astype(bool)

# --- 1. HadUK-Grid rainfall by police-recorded weather ---

summary = pd.DataFrame({
    "collisions": check.groupby(police_rain).size(),
    "median_rainfall_mm": check.groupby(police_rain)["rainfall_mm"].median(),
    "share_rain_day_pct": check.groupby(police_rain)["rainfall_mm"].apply(
        lambda s: 100 * (s >= rain_threshold).mean()
    ),
}).rename(index={True: "Police reported rain", False: "Police reported no rain"})

print("HadUK-Grid rainfall by police-recorded weather (aligned rain day):")
print(summary.round(2))

# --- 2. Did the 09:00 alignment improve agreement? ---

def hit_rate(rain_column):
    """Share of police-reported rain collisions that fall on a HadUK-Grid rain day."""
    return 100 * (check.loc[police_rain, rain_column] >= rain_threshold).mean()

def agreement(rain_column):
    """Share of collisions where police and HadUK-Grid agree on rain / no rain."""
    haduk_rain = check[rain_column] >= rain_threshold
    return 100 * (haduk_rain == police_rain).mean()

print(f"\nPolice-reported rain on a HadUK-Grid rain day (>= {rain_threshold} mm):")
print(f"  calendar-date match (Step 5): {hit_rate('rainfall_calendar_day'):.1f}%")
print(f"  aligned rain day (Step 5.2):  {hit_rate('rainfall_mm'):.1f}%")

print("Overall agreement between police report and HadUK-Grid:")
print(f"  calendar-date match (Step 5): {agreement('rainfall_calendar_day'):.1f}%")
print(f"  aligned rain day (Step 5.2):  {agreement('rainfall_mm'):.1f}%")

# --- 3. HadUK-Grid temperature by police-recorded road surface ---

known_surface = collisions["surface_ice_snow"].notna() & collisions["tas_c"].notna()
ice = collisions.loc[known_surface, "surface_ice_snow"].astype(bool)

print("\nMean temperature by police-recorded road surface:")
print(
    collisions.loc[known_surface]
    .groupby(ice)["tas_c"].agg(["count", "mean", "median"])
    .rename(index={True: "Ice or snow", False: "No ice or snow"})
    .round(2)
)

HadUK-Grid rainfall by police-recorded weather (aligned rain day):
                         collisions  median_rainfall_mm  share_rain_day_pct
police_reported_rain                                                       
Police reported no rain       84689                0.03                35.8
Police reported rain          11082                4.89                93.1

Police-reported rain on a HadUK-Grid rain day (>= 0.2 mm):
  calendar-date match (Step 5): 90.6%
  aligned rain day (Step 5.2):  93.1%
Overall agreement between police report and HadUK-Grid:
  calendar-date match (Step 5): 67.2%
  aligned rain day (Step 5.2):  67.5%

Mean temperature by police-recorded road surface:
                  count   mean  median
surface_ice_snow                      
No ice or snow    98140  11.86   11.94
Ice or snow        1407   1.91    1.71


#### Interpretation of Step 6

The HadUK-Grid values agree closely with the conditions police recorded at the scene, which confirms that collisions have been matched to the right cell and day.

| Police record | Collisions | HadUK-Grid result |
|---|---:|---|
| Rain reported | 11,082 | Median 4.89 mm; 93.1% on a rain day (≥ 0.2 mm) |
| No rain reported | 84,689 | Median 0.03 mm; 35.8% on a rain day |
| Ice or snow on the road | 1,407 | Mean 1.91 °C, median 1.71 °C |
| No ice or snow | 98,140 | Mean 11.86 °C, median 11.94 °C |

**Effect of the 09:00 alignment (Step 5.2).** The share of police-reported rain collisions falling on a HadUK-Grid rain day rose from 90.6% with calendar-date matching to 93.1% with the aligned rain day. The remaining mismatches fell from 9.4% to 6.9%, about a quarter fewer. Overall agreement changed little (67.2% to 67.5%), because it is dominated by the no-rain group.

**Why overall agreement is only 67.5%.** 35.8% of collisions with no rain reported still fell on a HadUK-Grid rain day. This is expected rather than an error: the police describe conditions at the moment of the collision, while HadUK-Grid gives a 24-hour total. On a day with a morning shower, an afternoon collision is correctly recorded as "fine" by the police and correctly counted as a rain day by HadUK-Grid. The two sources measure different things, which is why the analysis uses HadUK-Grid as an independent, consistent measure of daily weather.

**Temperature.** Collisions on icy or snowy roads occurred at a median daily mean temperature of 1.71 °C, about 10 °C colder than other collisions, as expected.

### Step 7: Build a Daily Summary for Great Britain

Comparing raw collision totals between rainy and dry conditions is misleading, because the number of each kind of day differs. A fair comparison needs collisions **per day** under each type of weather. This step builds a table with one row per day of 2025, combining the weather across Great Britain with the number of collisions that day.

**Weather for each day** (Great Britain cells only, from Step 3):
- mean rainfall across Great Britain (mm),
- share of Great Britain with a rain day (≥ 0.2 mm) and with a wet day (≥ 1 mm),
- mean temperature, and share of Great Britain with a mean temperature below 3 °C (near-freezing).

The British National Grid cells all cover the same area (5 km × 5 km), so a simple average over cells is also an area-weighted average.

**Collisions for each day:** total collisions, killed-or-seriously-injured (KSI) collisions, fatal collisions, casualties, KSI casualties, and collisions whose own cell had a rain day.

Collisions are grouped by their **rain day** from Step 5.2, so each day's collisions line up with that day's rainfall total. The temperature day runs midnight to midnight and overlaps the rain day by 15 of its 24 hours, which is close enough for a daily comparison and is noted as a limitation.

Day of the week, month and a weekend flag are added, because traffic, and so collisions, varies with them independently of the weather.

In [9]:
spatial_dims = ["projection_y_coordinate", "projection_x_coordinate"]
gb_rainfall = weather_gb["rainfall"]
gb_tas = weather_gb["tas"]

# --- 1. Weather across Great Britain for each day ---

# Averaging over the spatial dimensions skips the NaN cells outside
# Great Britain, so each value covers the 9,779 GB cells only.
# For the "share" columns, .where(gb_mask_da) restores NaN outside GB
# (a comparison such as >= 0.2 turns NaN into False, which would count as land).
daily_weather = pd.DataFrame({
    "gb_mean_rainfall_mm": gb_rainfall.mean(dim=spatial_dims).values,
    "gb_share_rain_day_pct": 100 * (gb_rainfall >= rain_threshold).where(gb_mask_da).mean(dim=spatial_dims).values,
    "gb_share_wet_day_pct": 100 * (gb_rainfall >= 1.0).where(gb_mask_da).mean(dim=spatial_dims).values,
    "gb_mean_tas_c": gb_tas.mean(dim=spatial_dims).values,
    "gb_share_below_3c_pct": 100 * (gb_tas < 3.0).where(gb_mask_da).mean(dim=spatial_dims).values,
}, index=pd.Index(weather_days, name="day"))

# --- 2. Collisions for each rain day ---

# KSI = killed or seriously injured, the standard UK road safety measure.
collisions["ksi"] = collisions["collision_severity"].isin([1, 2])

# Only collisions with rainfall attached can be placed on a rain day
# (this leaves out the 55 collisions belonging to the 31 December 2024 rain day).
rainfall_matched = collisions[collisions["rainfall_mm"].notna()].assign(
    ksi_casualties=lambda d: d["casualties_fatal"] + d["casualties_serious"]
)

daily_collisions = rainfall_matched.groupby("rain_day").agg(
    collisions=("collision_index", "size"),
    ksi_collisions=("ksi", "sum"),
    fatal_collisions=("collision_severity", lambda s: int((s == 1).sum())),
    casualties=("number_of_casualties", "sum"),
    ksi_casualties=("ksi_casualties", "sum"),
    collisions_rain_day_in_cell=("rainfall_mm", lambda s: int((s >= rain_threshold).sum())),
)
daily_collisions.index.name = "day"

# --- 3. Combine, and add calendar fields ---

# A left join keeps all 365 days, even any day with no collisions.
daily = daily_weather.join(daily_collisions, how="left")
count_cols = ["collisions", "ksi_collisions", "fatal_collisions",
              "casualties", "ksi_casualties", "collisions_rain_day_in_cell"]
daily[count_cols] = daily[count_cols].fillna(0).astype(int)

# Traffic varies by weekday and season regardless of the weather.
daily["day_of_week"] = daily.index.day_name()
daily["month"] = daily.index.month
daily["weekend"] = daily.index.dayofweek >= 5

# --- 4. Checks and summary ---

assert len(daily) == 365
assert daily["collisions"].sum() == len(rainfall_matched)

print("Daily table:", daily.shape)
print("Collisions in the daily table:", int(daily["collisions"].sum()))
print()
print(daily.describe().round(2).T[["mean", "min", "50%", "max"]])
daily.head()

Daily table: (365, 14)
Collisions in the daily table: 101468

                               mean     min     50%     max
gb_mean_rainfall_mm            2.92    0.00    1.49   22.89
gb_share_rain_day_pct         51.58    0.01   51.98  100.00
gb_share_wet_day_pct          37.90    0.00   31.66  100.00
gb_mean_tas_c                 10.15   -1.65   10.15   20.99
gb_share_below_3c_pct          9.59    0.00    0.00   98.80
collisions                   277.99  128.00  279.00  392.00
ksi_collisions                72.95   27.00   73.00  120.00
fatal_collisions               3.98    0.00    4.00   12.00
casualties                   350.14  176.00  351.00  483.00
ksi_casualties                80.20   31.00   80.00  140.00
collisions_rain_day_in_cell  118.05    0.00   94.00  383.00
month                          6.53    1.00    7.00   12.00


,gb_mean_rainfall_mm,gb_share_rain_day_pct,gb_share_wet_day_pct,gb_mean_tas_c,gb_share_below_3c_pct,collisions,ksi_collisions,fatal_collisions,casualties,ksi_casualties,collisions_rain_day_in_cell,day_of_week,month,weekend
day,,,,,,,,,,,,,,
2025-01-01,5.106195,97.433275,81.879538,5.882275,23.949279,138,32,3,178,38,135,Wednesday,1,False
2025-01-02,1.169275,40.505164,27.988547,0.514946,95.122201,246,66,2,318,73,26,Thursday,1,False
2025-01-03,0.936618,50.516413,26.444422,0.268366,97.596891,256,69,5,327,73,49,Friday,1,False
2025-01-04,9.910842,84.211065,76.122303,0.918108,82.073832,199,51,2,272,52,195,Saturday,1,True
2025-01-05,15.767877,87.861745,82.493097,2.113601,60.456079,177,52,3,227,59,177,Sunday,1,True


#### Interpretation of Step 7

The daily table covers all 365 days of 2025 and accounts for the 101,468 collisions with rainfall attached.

| Measure (per day) | Mean | Median | Range |
|---|---:|---:|---|
| Mean rainfall across Great Britain | 2.92 mm | 1.49 mm | 0.00 – 22.89 mm |
| Share of Great Britain with a rain day (≥ 0.2 mm) | 51.6% | 52.0% | 0.01% – 100% |
| Mean temperature across Great Britain | 10.15 °C | 10.15 °C | -1.65 – 20.99 °C |
| Share of Great Britain below 3 °C | 9.6% | 0% | 0% – 98.8% |
| Collisions | 278 | 279 | 128 – 392 |
| KSI collisions | 73 | 73 | 27 – 120 |
| Fatal collisions | 4.0 | 4 | 0 – 12 |

Daily weather varied widely across 2025, from completely dry days to days with rain across all of Great Britain, and from near-freezing days to warm summer days. This range is what makes a comparison of collisions per day under different weather possible.

Collision counts also vary for reasons unrelated to weather. The first rows show only 138 collisions on 1 January (a bank holiday) and fewer at weekends, when traffic is lighter. Some of those low-traffic days were also wet and cold, so the analysis must account for day of week, holidays and season before attributing differences to the weather.

### Step 8: Save the Integrated Datasets and Verify Them

Two files are saved for the analysis in notebook 04:

1. **`collisions_weather_2025.csv`**: one row per collision (101,523), with the cleaned collision fields from notebook 02 plus the matched weather: `rainfall_mm`, `tas_c`, `rain_day`, how the weather was matched (`weather_match`), the matched grid cell, and the coastal-cell flag. This is the project's final cleaned dataset.
2. **`daily_gb_weather_collisions_2025.csv`**: one row per day (365), with weather across Great Britain and daily collision counts.

Both are saved as CSV and reloaded, with column types specified again, to confirm nothing was lost or altered.

In [10]:
# Save both outputs to a folder for merged data.
output_dir = Path("../data/merged")
output_dir.mkdir(parents=True, exist_ok=True)

collisions_path = output_dir / "collisions_weather_2025.csv"
daily_path = output_dir / "daily_gb_weather_collisions_2025.csv"

# The collision index is a row number only, so it is not saved;
# the daily table keeps its index because it is the date.
collisions.to_csv(collisions_path, index=False)
daily.to_csv(daily_path)

for path in [collisions_path, daily_path]:
    print(f"Saved {path.name}: {path.stat().st_size / 1e6:.1f} MB")

# --- Reload and verify ---

# True/false columns are restored as pandas "boolean", which can hold blanks.
flag_columns_merged = flag_columns + ["coastal_cell", "ksi"]

collisions_reloaded = pd.read_csv(
    collisions_path,
    dtype={
        "collision_index": str,
        "local_authority_ons_district": str,
        **{column: "boolean" for column in flag_columns_merged},
    },
    parse_dates=["date", "datetime", "rain_day"],
)
daily_reloaded = pd.read_csv(daily_path, index_col="day", parse_dates=["day"])

# Collisions: same rows in the same order, same weather values and match labels.
# equal_nan=True treats blanks in the same position as equal.
assert collisions_reloaded.shape == collisions.shape
assert collisions_reloaded["collision_index"].equals(collisions["collision_index"])
assert np.allclose(collisions_reloaded["rainfall_mm"], collisions["rainfall_mm"], equal_nan=True)
assert np.allclose(collisions_reloaded["tas_c"], collisions["tas_c"], equal_nan=True)
assert collisions_reloaded["weather_match"].value_counts().equals(collisions["weather_match"].value_counts())

# Daily table: same size, same totals, same weather values.
assert daily_reloaded.shape == daily.shape
assert daily_reloaded["collisions"].sum() == daily["collisions"].sum()
assert np.allclose(daily_reloaded["gb_mean_rainfall_mm"], daily["gb_mean_rainfall_mm"])

print("\nMerged collisions:", collisions_reloaded.shape)
print("Daily summary:", daily_reloaded.shape)
print("All checks on the saved files passed.")

# Release the weather file now that everything needed has been extracted.
weather.close()

Saved collisions_weather_2025.csv: 28.2 MB
Saved daily_gb_weather_collisions_2025.csv: 0.0 MB

Merged collisions: (101523, 39)
Daily summary: (365, 14)
All checks on the saved files passed.
